In [ ]:
# !pip install -U pandas
# !pip install plotly
# !pip install nbformat

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
import plotly.express as px
import os


output_dir = "reports/torus"
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}")

def read_protein_data(file_path):
    """Read protein data from TSV file"""
    raw_data = pd.read_csv(file_path, delimiter="\t", header=None)
    raw_data.columns = ["name", "phi", "psi", "subtype"]
    return raw_data

def extract_protein_name(name: str) -> str:
    """Extract protein name from complex name field"""
    p_name = str(name).split(":")[0]
    return p_name.upper()

file_path = "fff/data/raw_data/torus/protein.tsv"
protein_data = read_protein_data(file_path)

# Protein Name extraction
protein_data['protein_name'] = protein_data['name'].apply(extract_protein_name)

print(f"Total data points: {len(protein_data)}")
print(f"Unique proteins: {protein_data['protein_name'].nunique()}")
print(f"Protein names: {protein_data['protein_name'].unique()[:10]}")  # First 10 Proteins

sample_data = protein_data 
# sample_data = protein_data.sample(10000)


In [1]:
def plot_protein_torus_matplotlib(data, max_proteins=100, title="Protein Data on Torus"):
    """
    Plot protein torsion angle data on a torus surface using Matplotlib.
    
    Visualizes (φ, ψ) angles from protein data mapped onto a 3D torus representation
    with flatter aspect ratio for better XY plane visualization.
    
    Args:
        data: DataFrame containing protein data with columns ['phi', 'psi', 'protein_name']
        max_proteins: Maximum number of unique proteins to display (top N by count)
        title: Plot title
        
    Returns:
        fig: Matplotlib figure object
        ax: Matplotlib axes object
    """
    
    # Limit number of proteins if too many
    unique_proteins = data['protein_name'].unique()
    if len(unique_proteins) > max_proteins:
        print(f"Too many proteins ({len(unique_proteins)}). Sampling {max_proteins} proteins...")
        protein_counts = data['protein_name'].value_counts()
        selected_proteins = protein_counts.head(max_proteins).index
        data = data[data['protein_name'].isin(selected_proteins)]
        unique_proteins = selected_proteins
    
    phi_degrees = data['phi'].values
    psi_degrees = data['psi'].values
    protein_names = data['protein_name'].values
    
    # Convert to radians
    phi_rad = np.deg2rad(phi_degrees)
    psi_rad = np.deg2rad(psi_degrees)
    
    # Torus parameters - flatter shape in XY plane
    R, r = 4.0, 0.8  # Major and minor radius
    
    # Convert to 3D coordinates
    x_data = (R + r * np.cos(phi_rad)) * np.cos(psi_rad)
    y_data = (R + r * np.cos(phi_rad)) * np.sin(psi_rad)
    z_data = r * np.sin(phi_rad)
    
    # Create color mapping for proteins
    unique_proteins = list(set(protein_names))
    color_map = plt.cm.tab20(np.linspace(0, 1, len(unique_proteins)))
    protein_to_color = {prot: color_map[i] for i, prot in enumerate(unique_proteins)}
    
    # Create figure
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # Create torus surface
    u = np.linspace(0, 2*np.pi, 40)
    v = np.linspace(0, 2*np.pi, 40)
    U, V = np.meshgrid(u, v)
    
    X = (R + r * np.cos(U)) * np.cos(V)
    Y = (R + r * np.cos(U)) * np.sin(V)
    Z = r * np.sin(U)
    
    ax.plot_wireframe(X, Y, Z, alpha=0.15, color='gray', rstride=3, cstride=3, linewidth=0.5)
    
    # Plot data points
    colors = [protein_to_color[prot] for prot in protein_names]
    scatter = ax.scatter(x_data, y_data, z_data,
                       c=colors, s=12, alpha=0.6,
                       edgecolors='black', linewidth=0.1)
    
    # Configure plot
    ax.set_xlabel('X', fontsize=12)
    ax.set_ylabel('Y', fontsize=12)
    ax.set_zlabel('Z', fontsize=12)
    ax.set_title(f"{title}\n({len(unique_proteins)} proteins, {len(data)} data points)", 
                fontsize=14, pad=20)
    
    # Set aspect ratio for flatter XY plane
    ax.set_box_aspect([2, 2, 1])
    
    # Set axis limits
    max_range = R + r + 1
    ax.set_xlim(-max_range, max_range)
    ax.set_ylim(-max_range, max_range)
    ax.set_zlim(-r-0.5, r+0.5)
    
    ax.grid(True, alpha=0.2)
    
    plt.tight_layout()
    return fig, ax


def plot_protein_torus_plotly(data, max_proteins=50, title="Protein Torsion Angles on Torus"):
    """
    Create interactive 3D torus plot of protein torsion angles using Plotly.
    
    Generates an interactive visualization with hover information showing
    protein details and angle values.
    
    Args:
        data: DataFrame with protein data columns
        max_proteins: Maximum number of proteins to display
        title: Plot title
        
    Returns:
        fig: Plotly figure object
    """
    
    # Limit number of proteins
    unique_proteins = data['protein_name'].unique()
    if len(unique_proteins) > max_proteins:
        protein_counts = data['protein_name'].value_counts()
        selected_proteins = protein_counts.head(max_proteins).index
        data = data[data['protein_name'].isin(selected_proteins)]
        unique_proteins = selected_proteins
    
    phi_degrees = data['phi'].values
    psi_degrees = data['psi'].values
    protein_names = data['protein_name'].values
    full_names = data['name'].values
    
    # Convert to 3D coordinates with flatter torus
    phi_rad = np.deg2rad(phi_degrees)
    psi_rad = np.deg2rad(psi_degrees)
    R, r = 4.0, 0.8  # Flatter torus parameters
    
    x_data = (R + r * np.cos(phi_rad)) * np.cos(psi_rad)
    y_data = (R + r * np.cos(phi_rad)) * np.sin(psi_rad)
    z_data = r * np.sin(phi_rad)
    
    # Create hover text
    hover_text = [
        f"Protein: {prot}<br>Residue: {name.split(':')[-1]}<br>φ: {phi:.1f}°<br>ψ: {psi:.1f}°"
        for prot, name, phi, psi in zip(protein_names, full_names, phi_degrees, psi_degrees)
    ]
    
    # Create figure
    fig = go.Figure()
    
    # Add data points
    fig.add_trace(go.Scatter3d(
        x=x_data,
        y=y_data, 
        z=z_data,
        mode='markers',
        marker=dict(
            size=3,
            opacity=0.7,
            color=phi_degrees,
            colorscale='Viridis',
            colorbar=dict(title="Phi (°)", x=1.1),
            line=dict(width=0.3, color='black')
        ),
        text=hover_text,
        hoverinfo='text',
        name='Protein Data',
        showlegend=False
    ))
    
    # Add torus surface
    u = np.linspace(0, 2*np.pi, 25)
    v = np.linspace(0, 2*np.pi, 25)
    U, V = np.meshgrid(u, v)
    
    X = (R + r * np.cos(U)) * np.cos(V)
    Y = (R + r * np.cos(U)) * np.sin(V)
    Z = r * np.sin(U)
    
    fig.add_trace(go.Surface(
        x=X, y=Y, z=Z,
        opacity=0.1,
        colorscale='Greys',
        showscale=False,
        hoverinfo='skip',
        name='Torus Surface',
        showlegend=False
    ))
    
    # Configure layout for flatter appearance
    fig.update_layout(
        title=dict(
            text=f"{title}<br><sup>{len(unique_proteins)} proteins, {len(data)} data points</sup>",
            x=0.5,
            xanchor='center',
            y=0.95
        ),
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y', 
            zaxis_title='Z',
            aspectratio=dict(x=2, y=2, z=1),  # Flatter XY aspect
            camera=dict(
                eye=dict(x=1.8, y=1.8, z=0.8),
                up=dict(x=0, y=0, z=1)
            ),
            xaxis=dict(range=[-R-r-1, R+r+1]),
            yaxis=dict(range=[-R-r-1, R+r+1]),
            zaxis=dict(range=[-r-1, r+1])
        ),
        width=1000,
        height=700,
        showlegend=False,
        margin=dict(l=0, r=0, b=0, t=100)
    )
    
    return fig

In [ ]:
def plot_proteins_separately(data, n_proteins=9, samples_per_protein=200):
    """
    Plot multiple proteins in separate subplots with individual torus visualizations.
    
    Creates a grid of 3D subplots showing torsion angle distributions for the top N
    most frequent proteins in the dataset.
    
    Args:
        data: DataFrame containing protein data with columns ['phi', 'psi', 'protein_name']
        n_proteins: Number of top proteins to display (default: 9)
        samples_per_protein: Maximum number of samples to show per protein (default: 200)
        
    Returns:
        fig: Matplotlib figure object with multiple subplots
    """
    
    # Select top N proteins by frequency
    top_proteins = data['protein_name'].value_counts().head(n_proteins).index
    
    # Calculate subplot grid dimensions
    n_cols = min(3, n_proteins)
    n_rows = (n_proteins + n_cols - 1) // n_cols
    
    # Create figure with appropriate size
    fig = plt.figure(figsize=(6*n_cols, 5*n_rows))
    
    # Torus parameters for flatter appearance
    R, r = 4.0, 0.8  # Major and minor radius
    
    # Create subplot for each protein
    for i, protein in enumerate(top_proteins):
        if i >= n_proteins:
            break
            
        ax = fig.add_subplot(n_rows, n_cols, i+1, projection='3d')
        
        # Get data for current protein
        protein_data = data[data['protein_name'] == protein]
        if len(protein_data) > samples_per_protein:
            protein_data = protein_data.sample(samples_per_protein)
        
        phi_degrees = protein_data['phi'].values
        psi_degrees = protein_data['psi'].values
        
        # Convert angles to radians
        phi_rad = np.deg2rad(phi_degrees)
        psi_rad = np.deg2rad(psi_degrees)
        
        # Convert to 3D torus coordinates
        x_data = (R + r * np.cos(phi_rad)) * np.cos(psi_rad)
        y_data = (R + r * np.cos(phi_rad)) * np.sin(psi_rad)
        z_data = r * np.sin(phi_rad)
        
        # Plot data points
        ax.scatter(x_data, y_data, z_data, s=25, alpha=0.8, color=f'C{i}', 
                  edgecolors='black', linewidth=0.3)
        
        # Create torus wireframe
        u = np.linspace(0, 2*np.pi, 20)
        v = np.linspace(0, 2*np.pi, 20)
        U, V = np.meshgrid(u, v)
        
        X = (R + r * np.cos(U)) * np.cos(V)
        Y = (R + r * np.cos(U)) * np.sin(V)
        Z = r * np.sin(U)
        
        ax.plot_wireframe(X, Y, Z, alpha=0.1, color='gray', linewidth=0.5)
        
        # Set subplot title and aspect ratio
        ax.set_title(f'{protein}\n({len(protein_data)} points)', fontsize=11)
        ax.set_box_aspect([2, 2, 1])  # Flatter XY aspect ratio
        
        # Set axis limits
        max_range = R + r + 1
        ax.set_xlim(-max_range, max_range)
        ax.set_ylim(-max_range, max_range)
        ax.set_zlim(-r-0.5, r+0.5)
        
        ax.grid(True, alpha=0.2)
    
    plt.tight_layout()
    return fig


In [ ]:
# Read and process protein data
file_path = "fff/data/raw_data/torus/protein.tsv"
protein_data = pd.read_csv(file_path, delimiter="\t", header=None)
protein_data.columns = ["name", "phi", "psi", "subtype"]
protein_data['protein_name'] = protein_data['name'].apply(lambda x: str(x).split(":")[0].upper())

print(f"Total data points: {len(protein_data)}")
print(f"Unique proteins: {protein_data['protein_name'].nunique()}")

# Create and save plots to reports/torus directory
print("\nCreating plots...")

# 1. Plot with all 494 proteins (Matplotlib)
fig1, ax1 = plot_protein_torus_matplotlib(protein_data, max_proteins=494)
plt.savefig(f'{output_dir}/protein_torus_all_proteins.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{output_dir}/protein_torus_all_proteins.pdf', bbox_inches='tight')
plt.show()

# 2. Interactive plot with all proteins (Plotly)
fig2 = plot_protein_torus_plotly(protein_data, max_proteins=494)
fig2.write_html(f"{output_dir}/protein_torus_interactive_all.html")
fig2.show()

# 3. Individual subplots for top 9 proteins
fig3 = plot_proteins_separately(protein_data, n_proteins=9, samples_per_protein=200)
plt.savefig(f'{output_dir}/protein_subplots_top9.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{output_dir}/protein_subplots_top9.pdf', bbox_inches='tight')
plt.show()

# 4. Simplified plot with top 20 proteins
fig4, ax4 = plot_protein_torus_matplotlib(protein_data, max_proteins=20, 
                                        title="Top 20 Proteins on Torus")
plt.savefig(f'{output_dir}/protein_torus_top20.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{output_dir}/protein_torus_top20.pdf', bbox_inches='tight')
plt.show()

print(f"\nAll plots saved to: {output_dir}/")
print("✓ protein_torus_all_proteins.png/.pdf")
print("✓ protein_torus_interactive_all.html") 
print("✓ protein_subplots_top9.png/.pdf")
print("✓ protein_torus_top20.png/.pdf")

In [ ]:
# Statistical analysis of protein data
print("\n=== Statistical Analysis ===")

# Number of data points per protein
protein_counts = protein_data['protein_name'].value_counts()
print(f"\nData points per protein (top 10):")
print(protein_counts.head(10))

# Angle statistics for each protein
angle_stats = protein_data.groupby('protein_name').agg({
    'phi': ['min', 'max', 'mean', 'std'],
    'psi': ['min', 'max', 'mean', 'std']
}).round(1)

print(f"\nAngle statistics per protein (first 5 proteins):")
print(angle_stats.head())

# Plot distribution of data points across proteins
plt.figure(figsize=(12, 6))
protein_counts.plot(kind='bar', color='skyblue', edgecolor='black', alpha=0.8)
plt.title('Distribution of Data Points per Protein', fontsize=14, pad=20)
plt.xlabel('Protein Name', fontsize=12)
plt.ylabel('Number of Data Points', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()

# Save the plot
plt.savefig(f'{output_dir}/data_points_distribution.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{output_dir}/data_points_distribution.pdf', bbox_inches='tight')
plt.show()

# Additional statistical summaries
print(f"\n=== Additional Statistics ===")
print(f"Total number of proteins: {protein_data['protein_name'].nunique()}")
print(f"Total data points: {len(protein_data)}")
print(f"Average data points per protein: {protein_data['protein_name'].value_counts().mean():.1f}")
print(f"Standard deviation of data points: {protein_data['protein_name'].value_counts().std():.1f}")
print(f"Minimum data points for a protein: {protein_data['protein_name'].value_counts().min()}")
print(f"Maximum data points for a protein: {protein_data['protein_name'].value_counts().max()}")

# Global angle statistics
print(f"\nGlobal angle statistics:")
print(f"Phi range: [{protein_data['phi'].min():.1f}, {protein_data['phi'].max():.1f}]")
print(f"Psi range: [{protein_data['psi'].min():.1f}, {protein_data['psi'].max():.1f}]")
print(f"Phi mean ± std: {protein_data['phi'].mean():.1f} ± {protein_data['phi'].std():.1f}")
print(f"Psi mean ± std: {protein_data['psi'].mean():.1f} ± {protein_data['psi'].std():.1f}")

# Save statistical results to CSV
protein_counts.to_csv(f'{output_dir}/protein_counts.csv')
angle_stats.to_csv(f'{output_dir}/angle_statistics.csv')

print(f"\nStatistical results saved to {output_dir}/")
print("✓ protein_counts.csv")
print("✓ angle_statistics.csv")
print("✓ data_points_distribution.png/.pdf")